# TP03: Perfilado de Datos
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 2: Herramientas en la Nube de Visualización de Datos

---

### 🎯 Objetivos del Trabajo Práctico

1. Utilizar herramientas de **profiling integradas** en Databricks
2. Reconocer **distribuciones iniciales** del dataset
3. Identificar **valores atípicos y anomalías**
4. Generar **estadísticas descriptivas** automáticas
5. Visualizar **correlaciones** entre variables

---

### 📁 Caso de Estudio: Perfilado de Ventas de Panadería

Utilizaremos las herramientas nativas de Databricks para perfilar y analizar los datos de ventas.

### 🕰️ Duración Estimada: 2 horas

In [0]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar estilo de visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías importadas correctamente")
print("\n🎨 Configuración de visualizaciones lista")

## Parte 1: Carga y Preparación de Datos

### 📂 Cargar los datasets de la panadería

Vamos a cargar los datasets y preparar un DataFrame consolidado para el perfilado.

In [0]:
# Ruta de los datasets
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

# Cargar todos los datasets
df_productos = pd.read_csv(ruta_datos + 'productos.csv')
df_sucursales = pd.read_csv(ruta_datos + 'sucursales.csv')
df_clientes = pd.read_csv(ruta_datos + 'clientes.csv')
df_ventas = pd.read_csv(ruta_datos + 'ventas.csv', parse_dates=['fecha'])
df_detalles_ventas = pd.read_csv(ruta_datos + 'detalles_ventas.csv')

print("✅ Todos los datasets cargados")
print(f"\n📊 Registros por dataset:")
print(f"  Productos: {len(df_productos):,}")
print(f"  Sucursales: {len(df_sucursales):,}")
print(f"  Clientes: {len(df_clientes):,}")
print(f"  Ventas: {len(df_ventas):,}")
print(f"  Detalles de ventas: {len(df_detalles_ventas):,}")

In [0]:
# Crear un dataset consolidado uniendo ventas con detalles y productos
df_consolidado = df_detalles_ventas.merge(
    df_productos[['producto_id', 'nombre', 'categoria', 'precio_unitario', 'costo_unitario']],
    on='producto_id',
    how='left',
    suffixes=('_detalle', '_producto')
).merge(
    df_ventas[['venta_id', 'fecha', 'sucursal_id', 'cliente_id', 'total']],
    on='venta_id',
    how='left'
)

# Calcular métricas adicionales
df_consolidado['ganancia'] = df_consolidado['subtotal'] - (df_consolidado['cantidad'] * df_consolidado['costo_unitario'])
df_consolidado['mes'] = df_consolidado['fecha'].dt.month
df_consolidado['dia_semana'] = df_consolidado['fecha'].dt.dayofweek
df_consolidado['es_fin_semana'] = df_consolidado['dia_semana'] >= 5

print("✅ Dataset consolidado creado")
print(f"\n📊 Total de registros: {len(df_consolidado):,}")
print(f"📌 Columnas: {len(df_consolidado.columns)}")
print(f"\n📄 Primeras filas:")
display(df_consolidado.head())

## Parte 2: Perfilado Estadístico Automático

### 📊 Análisis descriptivo completo

Vamos a analizar las características estadísticas de nuestros datos numéricos y categóricos.

In [0]:
# Seleccionar solo columnas numéricas
columnas_numericas = df_consolidado.select_dtypes(include=[np.number]).columns.tolist()

print("🔢 PERFIL DE VARIABLES NUMÉRICAS")
print("=" * 80)
print(f"\nVariables numéricas identificadas: {len(columnas_numericas)}")
print(f"Columnas: {columnas_numericas}")

print("\n📈 ESTADÍSTICAS DESCRIPTIVAS:")
print("=" * 80)

# Estadísticas descriptivas completas
estadisticas = df_consolidado[columnas_numericas].describe().T
estadisticas['missing'] = df_consolidado[columnas_numericas].isnull().sum()
estadisticas['missing_pct'] = (estadisticas['missing'] / len(df_consolidado) * 100).round(2)

display(estadisticas)

In [0]:
# Seleccionar columnas categóricas
columnas_categoricas = df_consolidado.select_dtypes(include=['object', 'bool']).columns.tolist()

print("🏷️ PERFIL DE VARIABLES CATEGÓRICAS")
print("=" * 80)

for col in columnas_categoricas[:5]:  # Mostrar las primeras 5
    valores_unicos = df_consolidado[col].nunique()
    valores_nulos = df_consolidado[col].isnull().sum()
    
    print(f"\n📌 {col}:")
    print(f"  Valores únicos: {valores_unicos}")
    print(f"  Valores nulos: {valores_nulos} ({valores_nulos/len(df_consolidado)*100:.2f}%)")
    print(f"  Top 5 valores más frecuentes:")
    print(df_consolidado[col].value_counts().head())

## Parte 3: Visualizaciones de Distribución

### 📉 Histogramas y boxplots

Las visualizaciones nos ayudan a entender la distribución de los datos y detectar valores atípicos.

In [0]:
# Visualizar la distribución de subtotales
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma
axes[0].hist(df_consolidado['subtotal'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('📉 Distribución de Subtotales', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Subtotal ($)')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(axis='y', alpha=0.3)

# Box plot
axes[1].boxplot(df_consolidado['subtotal'], vert=True)
axes[1].set_title('📦 Box Plot de Subtotales', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Subtotal ($)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Estadísticas de Subtotales:")
print(f"  Media: ${df_consolidado['subtotal'].mean():,.2f}")
print(f"  Mediana: ${df_consolidado['subtotal'].median():,.2f}")
print(f"  Desviación estándar: ${df_consolidado['subtotal'].std():,.2f}")

In [0]:
# Distribución de cantidades vendidas
fig, ax = plt.subplots(figsize=(12, 6))

df_consolidado['cantidad'].value_counts().sort_index().plot(kind='bar', ax=ax, color='skyblue', edgecolor='black')
ax.set_title('📐 Distribución de Cantidades Vendidas', fontsize=14, fontweight='bold')
ax.set_xlabel('Cantidad de unidades')
ax.set_ylabel('Frecuencia')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Estadísticas de Cantidades:")
print(df_consolidado['cantidad'].describe())

In [0]:
# Gráfico de barras: ventas por categoría
ventas_categoria = df_consolidado.groupby('categoria')['subtotal'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
ventas_categoria.plot(kind='barh', ax=ax, color='coral', edgecolor='black')
ax.set_title('🎨 Facturación Total por Categoría', fontsize=14, fontweight='bold')
ax.set_xlabel('Facturación Total ($)')
ax.set_ylabel('Categoría')
ax.grid(axis='x', alpha=0.3)

# Agregar valores en las barras
for i, v in enumerate(ventas_categoria):
    ax.text(v, i, f' ${v:,.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Parte 4: Análisis de Correlaciones

### 🔗 Relaciones entre variables

Vamos a identificar cómo se relacionan las diferentes variables numéricas.

In [0]:
# Seleccionar variables numéricas clave para analizar correlaciones
variables_analisis = ['cantidad', 'precio_unitario_detalle', 'descuento_porcentaje', 
                      'subtotal', 'precio_unitario_producto', 'costo_unitario', 'ganancia']

# Calcular matriz de correlación
matriz_correlacion = df_consolidado[variables_analisis].corr()

print("🔗 MATRIZ DE CORRELACIÓN")
print("=" * 80)
print(matriz_correlacion.round(3))

# Visualizar matriz de correlación
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matriz_correlacion, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, ax=ax, 
            cbar_kws={'label': 'Correlación'})
ax.set_title('🌡️ Mapa de Calor - Correlaciones', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [0]:
# Encontrar las correlaciones más fuertes (excluyendo la diagonal)
print("💪 CORRELACIONES MÁS FUERTES")
print("=" * 80)

# Convertir matriz a series y ordenar
correlaciones_ordenadas = matriz_correlacion.abs().unstack()
correlaciones_ordenadas = correlaciones_ordenadas[correlaciones_ordenadas < 1].sort_values(ascending=False)

print("\nTop 10 correlaciones más fuertes:")
for i, ((var1, var2), valor) in enumerate(correlaciones_ordenadas.head(10).items(), 1):
    print(f"{i:2d}. {var1:30s} <-> {var2:30s}: {valor:.3f}")

## Parte 5: Detección de Valores Atípicos (Outliers)

### 🔎 Identificar anomalías

Usaremos el método IQR (Rango Intercuartílico) para detectar valores atípicos.

In [0]:
# Función para detectar outliers usando IQR
def detectar_outliers(data, columna):
    Q1 = data[columna].quantile(0.25)
    Q3 = data[columna].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = data[(data[columna] < limite_inferior) | (data[columna] > limite_superior)]
    
    return outliers, limite_inferior, limite_superior

# Detectar outliers en subtotal
outliers_subtotal, lim_inf, lim_sup = detectar_outliers(df_consolidado, 'subtotal')

print("🔍 DETECCIÓN DE VALORES ATÍPICOS - SUBTOTAL")
print("=" * 80)
print(f"\nLímite inferior: ${lim_inf:,.2f}")
print(f"Límite superior: ${lim_sup:,.2f}")
print(f"\nNúmero de outliers detectados: {len(outliers_subtotal)} ({len(outliers_subtotal)/len(df_consolidado)*100:.2f}%)")

if len(outliers_subtotal) > 0:
    print("\n📊 Estadísticas de outliers:")
    print(outliers_subtotal['subtotal'].describe())
    print("\n📄 Primeros 10 outliers:")
    display(outliers_subtotal[['nombre', 'categoria', 'cantidad', 'subtotal']].head(10))

In [0]:
# Visualizar outliers con scatter plot
fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(df_consolidado.index, df_consolidado['subtotal'], 
           c='skyblue', alpha=0.5, s=10, label='Valores normales')
ax.scatter(outliers_subtotal.index, outliers_subtotal['subtotal'], 
           c='red', alpha=0.7, s=30, label='Outliers')

ax.axhline(y=lim_sup, color='orange', linestyle='--', linewidth=2, label=f'Límite superior (${lim_sup:,.0f})')
ax.axhline(y=lim_inf, color='green', linestyle='--', linewidth=2, label=f'Límite inferior (${lim_inf:,.0f})')

ax.set_title('🎯 Detección de Valores Atípicos en Subtotales', fontsize=14, fontweight='bold')
ax.set_xlabel('Indice de registro')
ax.set_ylabel('Subtotal ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Parte 6: Ejercicios Prácticos

### ✍️ Ejercicios para Resolver

#### **Ejercicio 1**: Análisis temporal
Crea un gráfico de líneas que muestre la evolución de las ventas totales por mes.

In [0]:
# EJERCICIO 1: Evolución temporal de ventas
# Pista: Agrupa por mes y suma los subtotales

# Tu código aquí:
ventas_mensuales = df_consolidado.groupby(df_consolidado['fecha'].dt.to_period('M'))['subtotal'].sum()

fig, ax = plt.subplots(figsize=(14, 6))
ventas_mensuales.plot(kind='line', ax=ax, marker='o', linewidth=2, markersize=8, color='steelblue')
ax.set_title('📈 Evolución de Ventas Mensuales', fontsize=14, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas Totales ($)')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"📊 Mes con mayor facturación: {ventas_mensuales.idxmax()} - ${ventas_mensuales.max():,.2f}")

## 🎯 Resumen del TP03

### ✅ Qué aprendimos:

1. **Perfilado estadístico**: Generamos estadísticas descriptivas automáticas
2. **Distribución de datos**: Visualizamos histogramas y box plots
3. **Análisis de categorías**: Comparamos ventas por categoría
4. **Correlaciones**: Identificamos relaciones entre variables
5. **Detección de outliers**: Usamos el método IQR para encontrar anomalías
6. **Visualizaciones**: Creamos gráficos con matplotlib y seaborn

### 💡 Insights clave:

* La mayoría de las ventas tienen subtotales moderados
* Existen algunos outliers que representan ventas grandes
* Ciertas categorías tienen mayor facturación
* Hay correlaciones fuertes entre cantidad y subtotal

### 🚀 Próximos pasos:

En el **TP04** aprenderemos a:
* Crear dashboards interactivos
* Usar visualizaciones avanzadas con plotly
* Combinar múltiples gráficos en un dashboard
* Presentar insights de forma profesional

---

**📝 Excelente trabajo! Ahora sabes cómo perfilar y visualizar datos de forma efectiva.**